In [1]:
!pip install jpegio

import os, torch, cv2, numpy as np
import torch.nn as nn
import torch.nn.functional as F
import jpegio as jio
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from scipy.ndimage import gaussian_filter

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f" Using Device: {DEVICE}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.5/73.5 MB 25.0 MB/s eta 0:00:0000:0100:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for jpegio: filename=jpegio-0.2.8-cp312-cp312-linux_x86_64.whl size=1277309 sha256=5291063c0b4fcf7f5896c059424c9ca1bc198212ef1a702d6c46697787e0b991
  Stored in directory: /root/.cache/pip/wheels/4e/5f/d9/3898c36bd7e9166a13a514d61f7ddd7310c192bfb443e758f8
Successfully built jpegio
 Using Device: cuda


In [2]:
import shutil
import os

# Working directory saaf karo
working_dir = '/kaggle/working'
for item in os.listdir(working_dir):
    item_path = os.path.join(working_dir, item)
    try:
        if os.path.isfile(item_path) or os.path.is_link(item_path):
            os.unlink(item_path)
        elif os.path.isdir(item_path):
            shutil.rmtree(item_path)
        print(f" Deleted: {item}")
    except Exception as e:
        print(f" Failed to delete {item_path}: {e}")

print(" Working directory Clean!")

 Failed to delete /kaggle/working/.virtual_documents: module 'posixpath' has no attribute 'is_link'
 Working directory Clean!


In [3]:
import os
import requests
import tarfile
from tqdm import tqdm

TAR_URL = "https://huggingface.co/datasets/plok5308/djpeg_dataset/resolve/main/jpeg_data.tar.gz"
EXTRACT_DIR = '/kaggle/working/extracted'

# Define exact limits per class (e.g., 30%, 10%, 10% of 50,000)
TRAIN_LIMIT = 171065 
VAL_LIMIT = 57022    
TEST_LIMIT = 57022   

# Create the nested directory structure
for split in ['train', 'val', 'test']:
    os.makedirs(os.path.join(EXTRACT_DIR, split, 'single'), exist_ok=True)
    os.makedirs(os.path.join(EXTRACT_DIR, split, 'double'), exist_ok=True)

def stream_extract_and_split(url, extract_dir, train_limit, val_limit, test_limit):
    # Track how many images we've saved to each split
    counts = {
        'single': {'train': 0, 'val': 0, 'test': 0},
        'double': {'train': 0, 'val': 0, 'test': 0}
    }
    
    print(" Streaming and splitting extraction...")
    
    response = requests.get(url, stream=True)
    with tarfile.open(fileobj=response.raw, mode="r|gz") as tar:
        for member in tqdm(tar, desc="Processing"):
            if member.isfile() and member.name.lower().endswith('.jpg'):
                
                # Identify the class
                img_class = 'single' if 'single' in member.name.lower() else ('double' if 'double' in member.name.lower() else None)
                
                if img_class:
                    target_split = None
                    
                    # Route to the correct folder based on what isn't full yet
                    if counts[img_class]['train'] < train_limit:
                        target_split = 'train'
                    elif counts[img_class]['val'] < val_limit:
                        target_split = 'val'
                    elif counts[img_class]['test'] < test_limit:
                        target_split = 'test'
                    
                    # Extract if it belongs to a split
                    if target_split:
                        counts[img_class][target_split] += 1
                        member.name = os.path.basename(member.name)
                        extract_path = os.path.join(extract_dir, target_split, img_class)
                        tar.extract(member, path=extract_path)
            
            # Check if all buckets for both classes are perfectly full
            single_done = all(counts['single'][s] == limit for s, limit in zip(['train', 'val', 'test'], [train_limit, val_limit, test_limit]))
            double_done = all(counts['double'][s] == limit for s, limit in zip(['train', 'val', 'test'], [train_limit, val_limit, test_limit]))
                           
            if single_done and double_done:
                print("\n All splits filled!")
                break

    print(f"\n Final Extraction Counts:")
    print(f"   Single -> Train: {counts['single']['train']} | Val: {counts['single']['val']} | Test: {counts['single']['test']}")
    print(f"   Double -> Train: {counts['double']['train']} | Val: {counts['double']['val']} | Test: {counts['double']['test']}")

stream_extract_and_split(TAR_URL, EXTRACT_DIR, TRAIN_LIMIT, VAL_LIMIT, TEST_LIMIT)

 Streaming and splitting extraction...


Processing: 0it [00:00, ?it/s]/tmp/ipykernel_55/3134220016.py:52: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extract(member, path=extract_path)
Processing: 855357it [05:40, 2509.66it/s]



 All splits filled!

 Final Extraction Counts:
   Single -> Train: 171065 | Val: 57022 | Test: 57022
   Double -> Train: 171065 | Val: 57022 | Test: 57022


In [4]:
import os, torch, cv2, numpy as np
import torch.nn as nn
import torch.nn.functional as F
import jpegio as jio
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from scipy.ndimage import gaussian_filter

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# DenseNet-121 Architecture: Growth rate k=32 [cite: 12, 219]
class DenseLayer(nn.Module):
    def __init__(self, in_ch, k):
        super().__init__()
        self.net = nn.Sequential(
            nn.BatchNorm2d(in_ch), nn.ReLU(inplace=True),
            nn.Conv2d(in_ch, 4*k, 1, bias=False),
            nn.BatchNorm2d(4*k), nn.ReLU(inplace=True),
            nn.Conv2d(4*k, k, 3, padding=1, bias=False)
        )
    def forward(self, x): 
        return torch.cat([x, self.net(x)], 1)

class DenseNet121_JPEG(nn.Module):
    def __init__(self, num_classes=2, k=32):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(2, 64, 7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True), 
            nn.MaxPool2d(3, stride=2, padding=1)
        )
        in_ch, config = 64, [6, 12, 24, 16] 
        self.blocks = nn.ModuleList()
        for i, nl in enumerate(config):
            block = nn.Sequential(*[DenseLayer(in_ch + j*k, k) for j in range(nl)])
            self.blocks.append(block)
            in_ch += nl * k
            if i < 3:
                trans = nn.Sequential(
                    nn.BatchNorm2d(in_ch), nn.ReLU(inplace=True),
                    nn.Conv2d(in_ch, in_ch//2, 1, bias=False), 
                    nn.AvgPool2d(2, stride=2)
                )
                self.blocks.append(trans)
                in_ch //= 2
        self.final_bn = nn.BatchNorm2d(in_ch)
        self.classifier = nn.Linear(in_ch, num_classes)

    def forward(self, x):
        x = self.stem(x)
        for b in self.blocks: x = b(x)
        x = F.relu(self.final_bn(x))
        x = F.adaptive_avg_pool2d(x, 1).flatten(1)
        return self.classifier(x)

In [9]:
def extract_features(path, patch_coords=None):
    try:
        jpg = jio.read(path)
        # 256x256 patch extraction [cite: 14, 280]
        if patch_coords:
            y, x = patch_coords
            dct = jpg.coef_arrays[0][y:y+256, x:x+256]
        else:
            dct = jpg.coef_arrays[0][:256, :256] 
        
        q = jpg.quant_tables[0].reshape(8,8) if jpg.quant_tables[0].ndim==1 else jpg.quant_tables[0]
        if dct.shape != (256, 256): return None

        b = 80 # Histogram range [-100, 100] [cite: 285, 329]
        bins = np.arange(-b, b+2) - 0.5
        H = np.zeros((64, 2*b+1), dtype=np.float32)
        Q = np.zeros((64, 2*b+1), dtype=np.float32)

        for idx, (u, v) in enumerate((u,v) for u in range(8) for v in range(8)):
            c = np.clip(dct[u::8, v::8].flatten(), -b, b)
            H[idx], _ = np.histogram(c, bins=bins)
            Q[idx] = float(q[u,v]) # Repeated q-factors [cite: 178]
            
        X = np.stack([H, Q], axis=-1) # Dimension 64 x 201 x 2 [cite: 185]
        X /= (X.max() + 1e-6)
        return X
    except: return None

class JPEGDataset(Dataset):
    def __init__(self, root_dir):
        self.file_list = []
        for label in ['single', 'double']:
            path = os.path.join(root_dir, label)
            if os.path.exists(path):
                for f in os.listdir(path):
                    self.file_list.append((os.path.join(path, f), 0 if label=='single' else 1))

    def __len__(self): return len(self.file_list)
    def __getitem__(self, idx):
        path, label = self.file_list[idx]
        X = extract_features(path)
        if X is None: return torch.zeros((2, 64, 201)), label
        return torch.tensor(X).permute(2, 0, 1), label

In [ ]:
from torch.utils.data import random_split

# Create datasets directly from the sub-folders
train_dataset = JPEGDataset(os.path.join(EXTRACT_DIR, 'train'))
val_dataset   = JPEGDataset(os.path.join(EXTRACT_DIR, 'val'))
test_dataset  = JPEGDataset(os.path.join(EXTRACT_DIR, 'test'))

# Load them straight into DataLoaders
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=64, shuffle=False)

# Setup
model = DenseNet121_JPEG().to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)
criterion = nn.CrossEntropyLoss()

best_val_acc = 0.0

print(f" Training Start (20 Epochs) | Train: len{train_dataset}, Val: len{val_dataset}...")
for epoch in range(20):
    if epoch == 15: # LR Scheduler
        for pg in optimizer.param_groups: pg['lr'] = 0.0005

    # ==========================
    # TRAINING PHASE
    # ==========================
    model.train()
    train_loss = 0.0
    tp, tn, fp, fn = 0, 0, 0, 0 
    
    for inputs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1} [Train]"):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        preds = torch.argmax(outputs, dim=1)
        
        # Calculate Training Metrics
        tp += ((preds == 1) & (labels == 1)).sum().item()
        tn += ((preds == 0) & (labels == 0)).sum().item()
        fp += ((preds == 1) & (labels == 0)).sum().item()
        fn += ((preds == 0) & (labels == 1)).sum().item()

    train_acc = (tp + tn) / (tp + tn + fp + fn + 1e-7)
    
    # ==========================
    # VALIDATION PHASE
    # ==========================
    model.eval()
    val_loss = 0.0
    v_tp, v_tn, v_fp, v_fn = 0, 0, 0, 0
    
    with torch.no_grad():
        for inputs, labels in tqdm(val_loader, desc=f"Epoch {epoch+1} [Val]"):
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            
            preds = torch.argmax(outputs, dim=1)
            
            # Calculate Validation Metrics
            v_tp += ((preds == 1) & (labels == 1)).sum().item()
            v_tn += ((preds == 0) & (labels == 0)).sum().item()
            v_fp += ((preds == 1) & (labels == 0)).sum().item()
            v_fn += ((preds == 0) & (labels == 1)).sum().item()

    # Paper Formulas for Validation
    val_acc = (v_tp + v_tn) / (v_tp + v_tn + v_fp + v_fn + 1e-7)
    val_tpr = v_tp / (v_tp + v_fn + 1e-7)
    val_tnr = v_tn / (v_tn + v_fp + 1e-7)
    
    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss = val_loss / len(val_loader)
    
    print(f" Epoch {epoch+1}")
    print(f"   Train | Loss: {avg_train_loss:.4f} | Acc: {train_acc*100:.2f}%")
    print(f"   Val   | Loss: {avg_val_loss:.4f} | Acc: {val_acc*100:.2f}% | TPR: {val_tpr*100:.2f}% | TNR: {val_tnr*100:.2f}%")
    
    # Save the model only if validation accuracy improves
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), '/kaggle/working/best_model.pth')
        print(f"New best model saved! (Val Acc: {best_val_acc*100:.2f}%)")

 Training Start (20 Epochs) | Train: len<__main__.JPEGDataset object at 0x7ceb70b65e20>, Val: len<__main__.JPEGDataset object at 0x7ceb518bc230>...


Epoch 1 [Val]: 100%|██████████| 1782/1782 [12:21<00:00,  2.40it/s]


 Epoch 1
   Train | Loss: 0.3178 | Acc: 85.17%
   Val   | Loss: 0.2591 | Acc: 88.63% | TPR: 81.93% | TNR: 95.34%
New best model saved! (Val Acc: 88.63%)


Epoch 2 [Val]: 100%|██████████| 1782/1782 [11:37<00:00,  2.55it/s]


 Epoch 2
   Train | Loss: 0.2520 | Acc: 89.08%
   Val   | Loss: 0.2358 | Acc: 90.05% | TPR: 83.28% | TNR: 96.81%
New best model saved! (Val Acc: 90.05%)


Epoch 3 [Val]: 100%|██████████| 1782/1782 [12:56<00:00,  2.29it/s]


 Epoch 3
   Train | Loss: 0.2264 | Acc: 90.41%
   Val   | Loss: 0.2294 | Acc: 90.39% | TPR: 84.25% | TNR: 96.53%
New best model saved! (Val Acc: 90.39%)


Epoch 4 [Val]: 100%|██████████| 1782/1782 [12:59<00:00,  2.28it/s]


 Epoch 4
   Train | Loss: 0.2106 | Acc: 91.18%
   Val   | Loss: 0.2107 | Acc: 91.07% | TPR: 88.51% | TNR: 93.62%
New best model saved! (Val Acc: 91.07%)


Epoch 5 [Train]:  73%|███████▎  | 3890/5346 [35:28<14:13,  1.71it/s]

In [ ]:
import os
print("🔎 Scanning every corner for your .pth file...")
!find / -name "*.pth" 2>/dev/null